⚠️ **Gemini Parse Error** — response could not be parsed as a valid notebook.
Raw output preserved below for manual recovery.

In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Migration: Employee File Processing Conversion\n",
        "**Source:** Employee ODI Script\n",
        "**Target:** Databricks Spark SQL / Delta Lake\n",
        "**Conversion Timestamp:** 2023-10-27"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"\")\n",
        "dbutils.widgets.text(\"ETL_PROC_WID\", \"\")\n",
        "dbutils.widgets.text(\"ODI_SESS_NO\", \"\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### ETL Parameters"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_etl_params AS\n",
        "SELECT \n",
        "    ${DATASOURCE_NUM_ID} AS datasource_num_id,\n",
        "    ${ETL_PROC_WID} AS etl_proc_wid,\n",
        "    '${ODI_SESS_NO}' AS odi_sess_no;"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "display(spark.sql(\"SELECT * FROM v_etl_params\"))"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### SCEN_TASK_NO {1}: OS File Listing\n",
        "Capture file names from the directory and write to a CSV file. Note: Windows local paths (C:\\\) must be mapped to Databricks Volumes or DBFS."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# MAGIC %sh\n",
        "# Converted from: dir /a /b C:\\\ODI\\\employee > C:\\\ODI\\\employeesfile\\name.csv\n",
        "# Assuming mount point /mnt/odi/ represents the legacy C:\\\ODI root\n",
        "ls -p /mnt/odi/employee | grep -v / > /mnt/odi/employeesfile/name.csv"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### SCEN_TASK_NO {2}: Truncate Log Table"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "TRUNCATE TABLE workspace.odi_trg.log_table;"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### SCEN_TASK_NO {3}: Reset Sequence"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "-- Converted from: ALTER_SEQUENCE ODI_TRG.file_sequence RESTART START WITH 1\n",
        "ALTER SEQUENCE workspace.odi_trg.file_sequence RESTART WITH 1;"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### Validation and Cleanup"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "SELECT 'log_table' AS table_name, COUNT(*) AS row_count FROM workspace.odi_trg.log_table;"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "**Manual Actions Required:**\n",
        "1. Ensure the directory path `/mnt/odi/employee` exists in Databricks File System (DBFS) or UC Volumes.\n",
        "2. Verify that `workspace.odi_trg.file_sequence` is defined as a Unity Catalog Sequence object."
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "codemirror_mode": {
        "name": "ipython",
        "version": 3
      },
      "file_extension": ".py",
      "mimetype": "text/x-python",
      "name": "python",
      "nbconvert_exporter": "python",
      "pygments_lexer": "ipython3",
      "version": "3.10.12"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}